In [25]:
from transformers import Trainer, T5Tokenizer, TrainingArguments, T5ForConditionalGeneration

In [26]:
import pandas as pd
train_data = pd.read_csv('datasets/samsum-train.csv')
val_data = pd.read_csv('datasets/samsum-validation.csv')

In [27]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [28]:
train_data.shape

(14732, 3)

In [29]:
val_data.shape

(818, 3)

In [30]:
train_data = train_data.sample(n=5000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [31]:
train_data.shape, val_data.shape

((5000, 3), (500, 3))

In [32]:
train_data.head()

,id,dialogue,summary
0,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
1,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
2,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
3,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
4,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."


In [33]:
import re
def clean_data(text):
    text = re.sub (r"\r\n"," ", text) # Lines
    text = re.sub(r"\s+",' ', text) # spaces
    text = re.sub(r"<.*?>", " " , text) # html tags <p> <h1>
    text = text.strip().lower()
    return text

train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

In [34]:
train_data.head()

,id,dialogue,summary
0,13811908,violet: hi! i came across this austin's articl...,violet sent claire austin's article.
1,13716431,pat: so does anyone know when the stream is go...,pat and lou are waiting for the stream but kev...
2,13810214,jane: jane: whaddya think? shona: this ur ti...,jane is updating her tinder profile tonight an...
3,13729823,"adam: do u have a map of paris? tom: yes, why?...",tom has a map of paris.
4,13681400,"frank: hi, how's the family? mike: great! sam'...","mike is happy, because sam's moved out. mike a..."


In [35]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [36]:
import torch
from torch.utils.data import Dataset

class SummaryDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        dialogue = self.data.iloc[index]['dialogue']
        summary = self.data.iloc[index]['summary']

        input_encoding = self.tokenizer(dialogue, padding='max_length', max_length=512, truncation=True, return_tensors = 'pt')
        target_encoding = self.tokenizer(summary, padding='max_length', max_length=150, truncation=True, return_tensors = 'pt')

        labels = target_encoding['input_ids'].squeeze()

        # Ignore padding tokens when calculating loss
        labels[labels == tokenizer.pad_token_id] = -100
        return {
            'input_ids':input_encoding['input_ids'].squeeze(),
            'attention_mask':input_encoding['attention_mask'].squeeze(),
            'labels':labels
        }
        

In [37]:
train_dataset = SummaryDataset(train_data, tokenizer)
val_dataset = SummaryDataset(val_data, tokenizer)

In [38]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [40]:
len(train_loader), len(val_loader)

(1250, 125)

In [41]:
# Device
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

# Model
model = T5ForConditionalGeneration.from_pretrained("t5-small")
model = model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

print("Device:", device)
print("Model device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Device: mps
Model device: mps:0


In [42]:
epochs = 4
for epoch in range(epochs):
    model.train()

    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        #remove old grad
        optimier.zero_grad()

        #forward pass
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        #calculate loss
        loss = output.loss


        #backpropagration
        loss.backward()

        #update weights
        optimier.step()


        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)

    print(f"epoch {epoch+1}/{epochs} training loss {avg_loss}")

epoch 1/4 training loss 2.674254540729523
epoch 2/4 training loss 2.6731341689109804
epoch 3/4 training loss 2.6758713742256166
epoch 4/4 training loss 2.679630228424072


In [ ]:
model.eval()

total_val_loss = 0

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        total_val_loss += loss.item()

average_val_loss = total_val_loss / len(val_loader)

print(
    f"Validation Loss: {average_val_loss:.4f}"
)